# Session 11: pandas Tables, Selecting and Filtering

Data Science with Python, Level 1, Stage 2: Working with Data

## Learning Objectives

By the end of this session you will be able to:

1. Create a DataFrame and load a CSV file.
2. Inspect a table with info, describe and value_counts.
3. Select columns and add calculated columns.
4. Filter rows with one or more conditions.
5. Select rows and columns together with loc.
6. Convert dates and filter by date.

## Where We Are

In Session 10 NumPy did fast maths on arrays of numbers. Real datasets mix numbers with names, dates and categories, and they arrive as tables with named columns. In Stage 1 you read a CSV file line by line and split each line yourself.

**pandas** is the most used data analysis library in Python. It loads a whole table in one line, describes it in a few more, and lets you ask questions such as "only the Campus branch" or "only big orders" with the same boolean masks you used on NumPy arrays. Today we explore a practice cafe's 1,000 orders.

In [ ]:
with open("menu.csv", "w") as f:
    f.write("item,price,sold\nLatte,3.5,128\nMuffin,3.0,64\n")

with open("menu.csv") as f:
    f.readline()
    for line in f:
        item, price, sold = line.strip().split(",")
        print(item, float(price) * int(sold))

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]

## 1. Series and DataFrames

A **Series** is a single column of values with an **index**, the labels of the rows. A **DataFrame** is a table of Series that share the same index. You can build a DataFrame from a dictionary of lists, where each key becomes a column name.

In [ ]:
students = pd.DataFrame({
    "name": ["Lina", "Omar", "Sara", "Hadi"],
    "city": ["Damascus", "Aleppo", "Damascus", "Homs"],
    "gpa": [3.6, 3.1, 3.9, 2.8],
})

students

## 2. Loading a CSV File

`pd.read_csv` reads a CSV file from your Colab folder or straight from a web address. Throughout the stage we use a practice dataset of **1,000 orders** from a fictional cafe with three branches, recorded over three months in 2025.

`head()` shows the first five rows; `tail()` shows the last five. Always look at your data first.

In [ ]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df.head()

## 3. Understanding the Table

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print(df["branch"].value_counts())
print(df["category"].nunique(), "categories:", list(df["category"].unique()))

## 4. Working with Columns

In [ ]:
df[["item", "unit_price", "quantity"]].head(3)

In [ ]:
df["revenue"] = df["unit_price"] * df["quantity"]

print(df["revenue"].sum().round(2))
df[["item", "unit_price", "quantity", "revenue"]].head(3)

## 5. Sorting and Ranking

In [ ]:
df.sort_values("revenue", ascending=False).head(3)[["date", "item", "quantity", "revenue"]]

## 6. Filtering Rows with Conditions

In [ ]:
mask = df["branch"] == "Campus"
print(mask.head())
print(mask.sum(), "orders at Campus")

campus = df[mask]
print(campus.shape)

In [ ]:
big = df[df["quantity"] >= 3]
print(len(big), "orders of 3 or more")
big[["date", "branch", "item", "quantity"]].head()

## 7. Combining Conditions

As with NumPy arrays, conditions are combined with `&` (and), `|` (or) and `~` (not). Each condition **must** be in its own brackets, because `&` and `|` are evaluated before comparisons.

In [ ]:
campus_big = df[(df["branch"] == "Campus") & (df["quantity"] >= 3)]
morning_or_card = df[(df["hour"] < 10) | (df["payment"] == "Card")]
not_coffee = df[~(df["category"] == "Coffee")]

print(len(campus_big), len(morning_or_card), len(not_coffee))

## 8. Selecting with loc and iloc

`loc` selects rows and columns **by label**: `df.loc[rows, columns]`. The rows can be a mask, so `loc` filters and picks columns in one step. `iloc` selects **by position**, like a list or a NumPy array: `df.iloc[0]` is the first row, whatever its label.

In [ ]:
df.loc[df["item"] == "Mocha", ["date", "branch", "quantity"]].head(3)

| You want | Use | Example |
| --- | --- | --- |
| Rows that match a condition | `loc` with a mask | `df.loc[df["hour"] < 10]` |
| Some columns of those rows | `loc[rows, cols]` | `df.loc[mask, ["item", "revenue"]]` |
| The first or last rows | `iloc` or `head` | `df.iloc[:5]`, `df.head()` |
| One cell by position | `iloc[row, col]` | `df.iloc[0, 3]` |

## 9. Useful Filters

In [ ]:
hot_drinks = df[df["item"].isin(["Latte", "Cappuccino", "Mocha"])]
lunch = df[df["hour"].between(12, 14)]
sandwiches = df[df["item"].str.contains("Sandwich")]

print(len(hot_drinks), len(lunch), len(sandwiches))
print(sandwiches["item"].unique().tolist())

## 10. Working with Dates

The `date` column was read as text. `pd.to_datetime` converts it into real dates. The `.dt` accessor then gives the parts of each date: `.dt.month`, `.dt.day_name()`, and more. Dates can also be compared directly with text such as `"2025-02-01"`.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.day_name()

print(df["date"].min(), df["date"].max())
print(df["month"].value_counts().sort_index())
df[["date", "month", "weekday"]].head(3)

## 11. Changing Selected Values

To change values in some rows, put the condition and the column inside **one** `loc`. Writing `df[mask]["col"] = value` looks similar but changes a temporary copy, so the original table does not change. In recent versions of pandas it silently does nothing.

In [ ]:
df["size"] = "regular"
df.loc[df["quantity"] >= 3, "size"] = "large"

print(df["size"].value_counts())

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| KeyError | A column name with a typo or wrong capitals | Check `df.columns`; names must match exactly. |
| Change lost | Calling `df.rename(...)` without storing the result | Write `df = df.rename(...)`. |
| ValueError | Using `and` or `or` between two conditions | Use `&` and `|`, with brackets around each condition. |
| Wrong rows | Forgetting the brackets: `df["a"] > 1 & df["b"] < 2` | Write `(df["a"] > 1) & (df["b"] < 2)`. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: First Look

Load `cafe_sales.csv`. Print its shape, the names of its columns, and the number of different items on the menu.

In [ ]:
# Write your code below

### In-Class Exercise 2: Busy Hours

Using the cafe data, find the three busiest hours of the day by number of orders.

In [ ]:
# Write your code below

### In-Class Exercise 3: Morning Coffee

Find the coffee orders placed before 10 o'clock. Which coffee is most popular in the morning?

In [ ]:
# Write your code below

### In-Class Exercise 4: Busiest Weekday

Convert the dates if needed, add a weekday column, and find which weekday has the most orders.

In [ ]:
# Write your code below

## Summary

- `pd.read_csv` loads a table; `head`, `info` and `describe` explain it.
- `value_counts` counts each category.
- `df[[...]]` selects columns; `df["new"] = ...` adds one.
- A condition gives a mask; `df[mask]` keeps the matching rows.
- Combine conditions with `&`, `|` and `~`, each in brackets.
- `loc[rows, columns]` filters and selects; `pd.to_datetime` makes real dates.

## What's Next

The cafe data was clean. Real data has missing values, duplicates, wrong types and typos. Next session you will find and fix them in a messy survey.